# 00 · Setup and overview
**Paper:** *Release-Stage Effects on AI Agent Answer Reliability Across Business Simulation Profiles* (target: IEEE IRAI 2027).

This notebook checks the environment. Run the notebooks in order; each one states what it verifies.

| Notebook | What it does | Needs Ollama? | Runtime |
|---|---|---|---|
| 01 | Regenerates the synthetic benchmark and checks it reproduces the frozen SHA-256 content hashes | no | < 1 min |
| 02 | Validates the scorer with scripted agents and worked examples (assertions) | no | ~1 min |
| 03 | Runs or loads the pilot; **re-scores every logged run from its trace** and checks it matches | only if `RUN_AGENT=True` | 1 min (load) / ≈ 9 h (run) |
| 04 | Pilot analysis: rates, contrasts, models, ablation, figure, paper macros | no | ~2 min |
| 05 | Simulation-based power analysis for the confirmatory design | no | ~10-20 min |
| 06 | Literature review: bibliography integrity, comparison matrix, novelty assessment | no | < 1 min |
| 07 | Paper consistency checks: citations resolve, numbers come from the analysis, banned phrases absent | no | < 1 min |


In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "harness"), str(ROOT / "benchmark")]
import pandas as pd
from IPython.display import Markdown, Image, display
pd.set_option("display.max_colwidth", 120)
print("repository root:", ROOT)

repository root: C:\IEEE-RESEARCH-WORK


In [2]:
import platform, importlib
print("python", platform.python_version())
for pkg in ["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "tabulate", "nbformat"]:
    print(f"{pkg:12s}", importlib.import_module(pkg).__version__)
print()
print(open(ROOT / "requirements.txt").read())

python 3.13.9
numpy        2.3.5
pandas       2.3.3
scipy        1.16.3
statsmodels  0.14.5
matplotlib   3.10.6
tabulate     0.9.0
nbformat     5.10.4

# Python 3.13.9 (Anaconda). Versions used to produce the results in this repository.
numpy==2.3.5
pandas==2.3.3
scipy==1.16.3
statsmodels==0.14.5
matplotlib==3.10.6
tabulate==0.9.0
pypdf==6.14.2
nbformat==5.10.4
nbconvert==7.16.6
ipykernel==6.31.0
# Agent runs only (optional): Ollama 0.40.13 with model qwen3:8b  ->  `ollama pull qwen3:8b`



In [3]:
# Optional: the local model server is only needed to RE-RUN the agent (notebook 03 with RUN_AGENT=True).
import urllib.request
try:
    tags = json.loads(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3).read())
    models = [m["name"] for m in tags["models"]]
    print("Ollama is running. Models:", models)
    print("qwen3:8b available:", any(m.startswith("qwen3:8b") for m in models))
except Exception as e:
    print("Ollama not reachable (fine unless you want to re-run agents):", e)

Ollama is running. Models: ['llama3.1-tools:8b', 'llama3.1:8b', 'qwen3:8b']
qwen3:8b available: True


In [4]:
import shutil
# list what the repository holds: skip files git ignores (local notes, build output, caches)
ignored = set(subprocess.run(["git", "ls-files", "-oi", "--exclude-standard", "--directory"], cwd=ROOT, capture_output=True,
                             text=True).stdout.splitlines()) if shutil.which("git") else set()
def tree(p, depth=0, max_depth=2):
    for c in sorted(p.iterdir(), key=lambda c: c.name.lower()):
        if c.name.startswith((".", "__")) or c.relative_to(ROOT).as_posix() + ("/" if c.is_dir() else "") in ignored:
            continue
        print("    " * depth + ("📁 " if c.is_dir() else "   ") + c.name)
        if c.is_dir() and depth < max_depth - 1 and c.name not in ("data",):
            tree(c, depth + 1, max_depth)
tree(ROOT)

📁 benchmark
    📁 data
       generate.py
📁 docs
    📁 audits
       CONFERENCE_REQUIREMENTS.md
    📁 templates
📁 harness
       analyze.py
       env.py
       llama3.1-tools.Modelfile
       run.py
       score.py
📁 literature
       novelty_assessment.md
       references.bib
       related_work.tex
       related_work_matrix.csv
       related_work_table.tex
       search_log.md
📁 notebooks
       00_setup_and_overview.ipynb
       01_benchmark_generation.ipynb
       02_scorer_validation.ipynb
       03_pilot_experiment.ipynb
       04_pilot_analysis.ipynb
       05_power_analysis.ipynb
       06_literature_review.ipynb
       07_paper_consistency_checks.ipynb
       build_notebooks.py
       run_inprocess.py
📁 paper
    📁 build
    📁 generated
       IEEEtran.bst
       IEEEtran.cls
       main.tex
       README.md
       references.bib
📁 pilot
       ablation_progress.log
    📁 analysis
    📁 analysis_llama
       analysis_llama_stdout.log
       analysis_stdout.log
    📁 discar